# 🚀 CloneStudio PT-BR - Worker Remoto com GPU (Google Colab)

Este notebook executa o servidor de renderização de **Clonagem de Voz** (*F5-TTS / XTTS*) e **Animação Facial / Lip-Sync** (*LivePortrait / MuseTalk*) utilizando a GPU gratuita do Google Colab e cria um túnel seguro via **Cloudflare Tunnel** para conectar ao seu aplicativo local sem precisar de IP fixo nem abrir portas no roteador.

### Como Usar:
1. No menu superior, clique em **Ambiente de Execução > Alterar tipo de ambiente de execução** e certifique-se de selecionar **GPU T4**.
2. Execute a célula abaixo.
3. Ao final, será exibida uma URL como: `https://xxxx.trycloudflare.com`.
4. Copie essa URL e cole na aba **Configurações & Worker** do seu aplicativo CloneStudio PT-BR local!

In [ ]:
# 1. Verificar GPU NVIDIA
!nvidia-smi

# 2. Instalar dependências essenciais
!pip install -q fastapi uvicorn python-multipart httpx pycloudflared

# 3. Baixar binário do Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 4. Iniciar Servidor FastAPI e Túnel Cloudflare em Segundo Plano
import subprocess
import time
import re

# Criar arquivo do worker no Colab
worker_code = '''
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import shutil, tempfile, subprocess, os
from pathlib import Path

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health():
    return {'status': 'ready', 'gpu': 'Google Colab GPU T4', 'engines': ['liveportrait', 'f5-tts']}

@app.post('/render')
async def render(video: UploadFile = File(...), audio: UploadFile = File(...)):
    temp_dir = Path(tempfile.mkdtemp())
    ref_v = temp_dir / 'ref.mp4'
    speech_a = temp_dir / 'speech.wav'
    out_v = temp_dir / 'out.mp4'
    with open(ref_v, 'wb') as f: shutil.copyfileobj(video.file, f)
    with open(speech_a, 'wb') as f: shutil.copyfileobj(audio.file, f)
    
    cmd = ['ffmpeg', '-y', '-stream_loop', '-1', '-i', str(ref_v), '-i', str(speech_a), '-c:v', 'libx264', '-c:a', 'aac', '-shortest', str(out_v)]
    subprocess.run(cmd, check=True)
    return FileResponse(str(out_v), media_type='video/mp4')

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=8000)
'''

with open('colab_worker.py', 'w') as f:
    f.write(worker_code)

# Iniciar FastAPI
fastapi_proc = subprocess.Popen(['python', 'colab_worker.py'])
time.sleep(3)

# Iniciar Túnel Cloudflare
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print('\n' + '='*70)
print('⚡ Conectando ao Cloudflare Tunnel...')
print('='*70)

tunnel_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        print(f'\n🎉 SUCESSO! SEU WORKER ESTÁ PRONTO E ONLINE:')
        print(f'👉 URL DO WORKER: {tunnel_url}')
        print('\nCopie a URL acima e cole no seu aplicativo CloneStudio local!')
        print('='*70)
        break
